In [16]:
# =====================================================================
# Project      : Banking Stress Testing Framework
# Notebook     : 01_Input_Validation.ipynb
# Purpose      : Import and validate the source portfolio CSV
# Input File   : portfolio_sample_10000.csv
# SAS Reference: 01_Input_Validation.sas
# =====================================================================


# =====================================================================
# Block 1: Mount Google Drive
# =====================================================================


from google.colab import drive

drive.mount("/content/drive", force_remount=True)


# =====================================================================
# Block 2: Import required Python libraries
# =====================================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("Libraries imported successfully.")
print("Pandas version:", pd.__version__)


# =====================================================================
# Block 3: Define file paths
# =====================================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Banking_Stress_Testing_V2_Python"
)

INPUT_FILE = PROJECT_ROOT / "data" / "portfolio_sample_10000.csv"
OUTPUT_DIR = PROJECT_ROOT / "output"
VALIDATION_DIR = PROJECT_ROOT / "validation"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Input file:", INPUT_FILE)
print("Input file exists:", INPUT_FILE.exists())

# =====================================================================
# Block 4: Import portfolio dataset
# Step 1: Import Portfolio Dataset
# =====================================================================


try:
    portfolio_raw = pd.read_csv(
        INPUT_FILE,
        low_memory=False
    )

    print("Portfolio dataset imported successfully.")
    print("Rows:", portfolio_raw.shape[0])
    print("Columns:", portfolio_raw.shape[1])

except FileNotFoundError:
    raise FileNotFoundError(
        f"Input CSV was not found: {INPUT_FILE}"
    )

except pd.errors.EmptyDataError:
    raise ValueError(
        f"Input CSV contains no data: {INPUT_FILE}"
    )

except pd.errors.ParserError as error:
    raise ValueError(
        f"Input CSV could not be parsed: {error}"
    )



# ---------------------------------------------------------------------
# Block 4: Import portfolio dataset
# Step 2: Dataset Structure Validation
# ---------------------------------------------------------------------

structure_validation = pd.DataFrame({
    "Variable_Name": portfolio_raw.columns,
    "Python_Data_Type": [
        str(data_type)
        for data_type in portfolio_raw.dtypes
    ],
    "Non_Missing_Count": [
        portfolio_raw[column].notna().sum()
        for column in portfolio_raw.columns
    ],
    "Missing_Count": [
        portfolio_raw[column].isna().sum()
        for column in portfolio_raw.columns
    ],
    "Unique_Values": [
        portfolio_raw[column].nunique(dropna=True)
        for column in portfolio_raw.columns
    ]
})

display(structure_validation)

structure_validation.to_csv(
    VALIDATION_DIR / "01_dataset_structure_validation.csv",
    index=False
)



# ---------------------------------------------------------------------
# Block 4: Import portfolio dataset
# Step 3: Record Count Validation
# ---------------------------------------------------------------------

row_count = len(portfolio_raw)

record_count_validation = pd.DataFrame({
    "Validation_Metric": ["Portfolio_Row_Count"],
    "Result": [row_count],
    "Expected_Result": [10000],
    "Status": [
        "PASS" if row_count == 10000 else "FAIL"
    ]
})

display(record_count_validation)

record_count_validation.to_csv(
    VALIDATION_DIR / "02_record_count_validation.csv",
    index=False
)


# ---------------------------------------------------------------------
# Block 4: Import portfolio dataset
# Step 4: Numeric Variable Validation
# ---------------------------------------------------------------------

numeric_columns = [
    "Outstanding_Balance",
    "Credit_Score",
    "Previous_Credit_Score",
    "Current_DPD",
    "Previous_DPD",
    "Interest_Rate",
    "Loan_Term_Months",
    "Remaining_Term_Months",
    "Collateral_Value"
]

numeric_validation_rows = []

for column in numeric_columns:

    numeric_series = pd.to_numeric(
        portfolio_raw[column],
        errors="coerce"
    )

    numeric_validation_rows.append({
        "Variable": column,
        "N": int(numeric_series.notna().sum()),
        "NMISS": int(numeric_series.isna().sum()),
        "Mean": numeric_series.mean(),
        "Median": numeric_series.median(),
        "Std_Dev": numeric_series.std(ddof=1),
        "Minimum": numeric_series.min(),
        "P25": numeric_series.quantile(0.25),
        "P75": numeric_series.quantile(0.75),
        "Maximum": numeric_series.max()
    })

numeric_validation = pd.DataFrame(
    numeric_validation_rows
)

display(numeric_validation)

numeric_validation.to_csv(
    VALIDATION_DIR / "03_numeric_variable_validation.csv",
    index=False
)


# ---------------------------------------------------------------------
# Block 4: Import portfolio dataset
# Step 5: Categorical Variable Validation
# ---------------------------------------------------------------------


categorical_columns = [
    "Customer_Segment",
    "Product_Type",
    "Region"
]

categorical_results = []

for column in categorical_columns:

    frequency_table = (
        portfolio_raw[column]
        .fillna("<MISSING>")
        .value_counts(dropna=False)
        .rename_axis("Category")
        .reset_index(name="Frequency")
    )

    frequency_table.insert(
        0,
        "Variable",
        column
    )

    frequency_table["Percent"] = (
        frequency_table["Frequency"]
        / len(portfolio_raw)
        * 100
    )

    categorical_results.append(frequency_table)

categorical_validation = pd.concat(
    categorical_results,
    ignore_index=True
)

display(categorical_validation)

categorical_validation.to_csv(
    VALIDATION_DIR / "04_categorical_variable_validation.csv",
    index=False
)


# ---------------------------------------------------------------------
# Block 4: Import portfolio dataset
# Step 6: Data Quality Validation
# ---------------------------------------------------------------------

data_quality_summary = pd.DataFrame({
    "Metric": [
        "Total_Rows",
        "Unique_Customers",
        "Unique_Loans",
        "Missing_Credit_Scores",
        "Missing_Current_DPD",
        "Missing_Collateral_Values"
    ],
    "Result": [
        len(portfolio_raw),
        portfolio_raw["Customer_ID"].nunique(dropna=True),
        portfolio_raw["Loan_ID"].nunique(dropna=True),
        portfolio_raw["Credit_Score"].isna().sum(),
        portfolio_raw["Current_DPD"].isna().sum(),
        portfolio_raw["Collateral_Value"].isna().sum()
    ]
})

display(data_quality_summary)

data_quality_summary.to_csv(
    VALIDATION_DIR / "05_data_quality_summary.csv",
    index=False
)



# ---------------------------------------------------------------------
# Block 4: Import portfolio dataset
# Step 7: Secured Loan Collateral Validation
# ---------------------------------------------------------------------

secured_product_names = [
    "MORTGAGE",
    "AUTO LOAN"
]

normalized_product_type = (
    portfolio_raw["Product_Type"]
    .astype("string")
    .str.strip()
    .str.upper()
)

secured_portfolio = portfolio_raw.loc[
    normalized_product_type.isin(
        secured_product_names
    )
].copy()

secured_accounts = len(secured_portfolio)

missing_secured_collateral = (
    secured_portfolio["Collateral_Value"]
    .isna()
    .sum()
)

secured_collateral_validation = pd.DataFrame({
    "Metric": [
        "Secured_Accounts",
        "Missing_Collateral"
    ],
    "Result": [
        secured_accounts,
        missing_secured_collateral
    ]
})

display(secured_collateral_validation)

secured_collateral_validation.to_csv(
    VALIDATION_DIR /
    "06_secured_collateral_validation.csv",
    index=False
)



# ---------------------------------------------------------------------
# Block 4: Import portfolio dataset
# Step 8: Save Raw Portfolio Output
# ---------------------------------------------------------------------

RAW_OUTPUT_FILE = (
    OUTPUT_DIR / "portfolio_raw.csv"
)

portfolio_raw.to_csv(
    RAW_OUTPUT_FILE,
    index=False
)

print(
    "Raw portfolio output created:",
    RAW_OUTPUT_FILE
)

print(
    "Output records:",
    len(portfolio_raw)
)


# ---------------------------------------------------------------------
# Step 9: Final Phase 01 Validation Summary
# ---------------------------------------------------------------------

expected_results = {
    "Portfolio_Records": 10000,
    "Unique_Customers": 10000,
    "Unique_Loans": 10000,
    "Missing_Credit_Scores": 100,
    "Missing_Current_DPD": 75,
    "Missing_Secured_Collateral": 40
}

python_results = {
    "Portfolio_Records": len(portfolio_raw),
    "Unique_Customers":
        portfolio_raw["Customer_ID"].nunique(),
    "Unique_Loans":
        portfolio_raw["Loan_ID"].nunique(),
    "Missing_Credit_Scores":
        int(portfolio_raw["Credit_Score"].isna().sum()),
    "Missing_Current_DPD":
        int(portfolio_raw["Current_DPD"].isna().sum()),
    "Missing_Secured_Collateral":
        int(missing_secured_collateral)
}

reconciliation_rows = []

for metric, expected_value in expected_results.items():

    python_value = python_results[metric]

    reconciliation_rows.append({
        "Metric": metric,
        "SAS_Expected_Result": expected_value,
        "Python_Result": python_value,
        "Difference": python_value - expected_value,
        "Status": (
            "PASS"
            if python_value == expected_value
            else "REVIEW"
        )
    })

phase_01_reconciliation = pd.DataFrame(
    reconciliation_rows
)

display(phase_01_reconciliation)

phase_01_reconciliation.to_csv(
    VALIDATION_DIR /
    "07_sas_python_phase01_reconciliation.csv",
    index=False
)



# ---------------------------------------------------------------------
# Step 10: Completion Message
# ---------------------------------------------------------------------

print("=" * 60)
print("PYTHON V2 - MODULE 01 COMPLETED")
print("INPUT VALIDATION EXECUTED SUCCESSFULLY")
print("SOURCE RECORDS:", len(portfolio_raw))
print("RAW OUTPUT:", RAW_OUTPUT_FILE)
print("VALIDATION OUTPUT:", VALIDATION_DIR)
print("=" * 60)

Mounted at /content/drive
Libraries imported successfully.
Pandas version: 2.2.3
Project root: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python
Input file: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/data/portfolio_sample_10000.csv
Input file exists: True
Portfolio dataset imported successfully.
Rows: 10000
Columns: 15


,Variable_Name,Python_Data_Type,Non_Missing_Count,Missing_Count,Unique_Values
0,Customer_ID,int64,10000,0,10000
1,Loan_ID,int64,10000,0,10000
2,Customer_Segment,object,10000,0,3
3,Product_Type,object,10000,0,6
4,Outstanding_Balance,float64,10000,0,10000
5,Credit_Score,float64,9900,100,413
6,Previous_Credit_Score,int64,10000,0,399
7,Current_DPD,float64,9925,75,9
8,Previous_DPD,int64,10000,0,15
9,Interest_Rate,float64,10000,0,2286


,Validation_Metric,Result,Expected_Result,Status
0,Portfolio_Row_Count,10000,10000,PASS


,Variable,N,NMISS,Mean,Median,Std_Dev,Minimum,P25,P75,Maximum
0,Outstanding_Balance,10000,0,4.758191e+06,1840188.745,8.942135e+06,5688.98,6.134422e+05,5.942842e+06,74748873.88
1,Credit_Score,9900,100,6.999426e+02,700.000,7.283058e+01,375.00,6.500000e+02,7.500000e+02,900.00
2,Previous_Credit_Score,10000,0,7.000260e+02,700.000,6.947173e+01,402.00,6.520000e+02,7.470000e+02,900.00
3,Current_DPD,9925,75,8.451385e+00,0.000,2.606672e+01,0.00,0.000000e+00,0.000000e+00,180.00
4,Previous_DPD,10000,0,1.340100e+01,0.000,2.671750e+01,0.00,0.000000e+00,1.500000e+01,210.00
5,Interest_Rate,10000,0,1.505848e+01,11.975,8.575961e+00,7.00,9.480000e+00,1.653000e+01,41.97
6,Loan_Term_Months,10000,0,1.003122e+02,61.000,9.162385e+01,12.00,3.800000e+01,1.342500e+02,360.00
7,Remaining_Term_Months,10000,0,5.039070e+01,29.000,6.012038e+01,1.00,1.300000e+01,5.900000e+01,347.00
8,Collateral_Value,4598,5402,4.861629e+06,2916707.405,4.357009e+06,144674.80,1.455447e+06,7.866432e+06,18430839.12


,Variable,Category,Frequency,Percent
0,Customer_Segment,Retail,7715,77.15
1,Customer_Segment,SME,1802,18.02
2,Customer_Segment,Corporate,483,4.83
3,Product_Type,Mortgage,2550,25.50
4,Product_Type,Personal Loan,2117,21.17
5,Product_Type,Auto Loan,2088,20.88
6,Product_Type,Credit Card,1555,15.55
7,Product_Type,SME Loan,1285,12.85
8,Product_Type,Corporate Loan,405,4.05
9,Region,West,3179,31.79


,Metric,Result
0,Total_Rows,10000
1,Unique_Customers,10000
2,Unique_Loans,10000
3,Missing_Credit_Scores,100
4,Missing_Current_DPD,75
5,Missing_Collateral_Values,5402


,Metric,Result
0,Secured_Accounts,4638
1,Missing_Collateral,40


Raw portfolio output created: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/output/portfolio_raw.csv
Output records: 10000


,Metric,SAS_Expected_Result,Python_Result,Difference,Status
0,Portfolio_Records,10000,10000,0,PASS
1,Unique_Customers,10000,10000,0,PASS
2,Unique_Loans,10000,10000,0,PASS
3,Missing_Credit_Scores,100,100,0,PASS
4,Missing_Current_DPD,75,75,0,PASS
5,Missing_Secured_Collateral,40,40,0,PASS


PYTHON V2 - MODULE 01 COMPLETED
INPUT VALIDATION EXECUTED SUCCESSFULLY
SOURCE RECORDS: 10000
RAW OUTPUT: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/output/portfolio_raw.csv
VALIDATION OUTPUT: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/validation
